# Task 2 — analyse the Kaggle results and choose a submission (local, no training)

1. Put the downloaded `task2_results.zip` in this folder (`Question 2 - Leaderboard/`).
2. Run the cells top to bottom. Nothing here trains a model or submits anything.

**Two kinds of submission candidate per experiment**
* `<experiment>__checkpoint` — the average of the experiment's validation models, forecasting from the end of history. Its validation estimate (`ens_RMSE`) is exact: these are the very models that were scored. Declared E = validation epochs run.
* `<experiment>__refit` — the same seeds refit on all history for their best epoch count (top experiments only). Uses ~26 more weeks of data (including the most recent ones), but has **no validation score of its own**; the number shown is borrowed from its checkpoint twin. Declared E = validation epochs + refit epochs.

In [ ]:
import zipfile
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import task2_experiments as tx

RESULTS_ROOT = Path("task2_results")
if not RESULTS_ROOT.exists():
    with zipfile.ZipFile("task2_results.zip") as archive:
        archive.extractall(RESULTS_ROOT)
tx.configure(RESULTS_ROOT)
pd.set_option("display.width", 220, "display.max_columns", 40, "display.precision", 2)

## 1. Did the session complete?

In [ ]:
import json
log = json.loads((RESULTS_ROOT / "plan_log.json").read_text())
for event in log["events"]:
    print(f"{event['t_hours']:6.2f} h  {event['message']}")
print("\nfailures:", [f["step"] for f in log["failures"]] or "none")

## 2. Experiments: mean ± std over seeds, and the seed-ensemble score

In [ ]:
summary = tx.compare()
summary[["experiment", "seeds", "params", "RMSE_mean", "RMSE_std", "recent_RMSE_mean",
         "recent_RMSE_std", "ens_RMSE", "ens_recent_RMSE", "MAE_mean", "sMAPE_mean",
         "best_epoch_mean", "notes"]]

In [ ]:
s = summary.sort_values("RMSE_mean", ascending=False)
fig, ax = plt.subplots(figsize=(8, 0.45 * len(s) + 1))
ax.errorbar(s.RMSE_mean, s.experiment, xerr=s.RMSE_std, fmt="o", capsize=3,
            label="mean ± std over seeds")
ax.plot(s.ens_RMSE, s.experiment, "x", label="average of seed forecasts")
ax.set_xlabel("validation RMSE (176 blocks)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / "comparison.png", dpi=150)
plt.show()

## 3. Every individual run (per-seed tables for the report)

In [ ]:
tx.results()[["experiment", "seed", "params", "best_epoch", "epochs_run", "RMSE", "MAE",
              "sMAPE", "recent_RMSE", "raw_negative_count", "train_seconds"]]

## 4. External-data ablation (required in the report)
`E0_baseline` vs `E0_none`: same architecture, with and without the optional file, across seeds.

In [ ]:
tx.compare(["E0_baseline", "E0_none"])[["experiment", "seeds", "RMSE_mean", "RMSE_std",
                                         "MAE_mean", "sMAPE_mean", "recent_RMSE_mean"]]

## 5. Submission candidates
Decision rule (decide before looking at the leaderboard): prefer the candidate with the lowest validation `ens_RMSE`, check that `ens_recent_RMSE` agrees, and only switch away from attempt 1's design if the gain is larger than the seed spread (`RMSE_std`) of the experiments involved.

In [ ]:
candidates = tx.submissions().sort_values(["ens_RMSE", "kind"])
candidates[["label", "kind", "members", "ens_RMSE", "ens_recent_RMSE", "ens_MAE", "ens_sMAPE",
            "declared_parameters_P", "declared_epochs_E", "forecast_mean", "forecast_min",
            "forecast_max", "zeros"]]

In [ ]:
CHOSEN = candidates.label.iloc[0]   # or type a label, e.g. "E4_anchor__refit"
chosen_dir = RESULTS_ROOT / "submissions" / CHOSEN
info = json.loads((chosen_dir / "submission.json").read_text())
forecast = pd.read_csv(chosen_dir / "forecast.csv")
train = pd.read_csv(tx.af.DATA / "student_train.csv")

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(train.time_idx[-336:], train.value[-336:], label="history (last 336 steps)")
for label in candidates.label.iloc[:3]:
    other = pd.read_csv(RESULTS_ROOT / "submissions" / label / "forecast.csv")
    ax.plot(other.time_idx, other.value, lw=1, alpha=0.6, label=label)
ax.plot(forecast.time_idx, forecast.value, lw=2, label=f"CHOSEN: {CHOSEN}")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

assert len(forecast) == 168 and forecast.time_idx.tolist() == list(range(43657, 43825))
print(f"{CHOSEN}: declare P = {info['declared_parameters_P']}   E = {info['declared_epochs_E']}")
print("members:", info["per_member"])
print()
print((chosen_dir / "predictions.txt").read_text())

## 6. Leaderboard log — record what each attempt actually contained

In [ ]:
tx.record_leaderboard(1, "direct_level_s1 validation checkpoint", 84.4326, 56.1390, 82.55, 30819, 15)
tx.record_leaderboard(2, "likely final_all_s1 baseline refit (identical to 3)", 85.4256, 60.9015,
                      76.19, 30625, None, note="contents and declared E not confirmed")
log = tx.record_leaderboard(3, "final_all_s1 baseline refit", 85.4256, 60.9015, 76.19, 30625,
                            None, note="fill in the declared E you entered")
# After attempt 4, fill in the returned metrics:
# log = tx.record_leaderboard(4, CHOSEN, rmse, mae, smape,
#                             info["declared_parameters_P"], info["declared_epochs_E"])
log